# U-net

U-Net is a convolutional neural network architecture designed for biomedical image segmentation, characterized by a symmetric "U"-shaped encoder-decoder structure with skip connections that preserve spatial information for precise pixel-wise classification.

output size = (input size - conv kernal size + 2 * paddig) /  stride + 1

<img src="U_net_img1.png" width="800" />

In [ ]:
import torch
import torch.nn as nn

class BasicConvBlock(nn.Module):
    """
    basic convoluntional block: Conv2d -> BatchNorm 
    """
    def __init__(self, in_channels, out_channels, num_convs=2, use_bn=True):
        """
        parameters:
            in_channels: input channel
            out_channels: output channel
            num_convs:number of convluntional layer
            use_bn: use batch norm T/F
        """
        super().__init__()
        
        layers = []
        # first layer: input channel -> output channel
        layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
        if use_bn:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU(inplace=True))
        
        # Remaining convolutional layers: output channels → output channels (keeping the number of channels unchanged)
        for _ in range(num_convs - 1):
            layers.append(nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1))
            if use_bn:
                layers.append(nn.BatchNorm2d(out_channels))
            layers.append(nn.ReLU(inplace=True))
        
        self.conv_block = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.conv_block(x)

Downsample

This block is located on the left side of the Unet (encoder), and its function is: to continuously reduce the size of the image and extract features over a larger area (just like continuously shrinking a map to observe the more macroscopic structure).

In [ ]:
class DownSample(nn.Module):
    """
    Downsampling block: DoubleConv -> MaxPool2d     
    Function: 
    1. Firstly, use DoubleConv to extract the features at the current scale.
    2. Then, use MaxPool2d to reduce the image size by half.
    3. At the same time, return two results: 
       - down: The extracted feature map (used for subsequent "skip connections")
       - p: The pooled feature map (passed to the next layer for further downsampling)
    """
    def __init__(self, in_channels, out_channels):
        super(DownSample, self).__init__()
        
        # First, perform double convolution to extract features
        self.conv = DoubleConv(in_channels, out_channels)
        
        # Perform maximum pooling to reduce the size (kernel_size=2, stride=2)
        # Result: 256x256 -> 128x128
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
    
    def forward(self, x):
        """
        Forward pass:
           Input x: [batch, in_channels, height, width]             
    Return:
        down: [batch, out_channels, height, width] (extracted feature map)
        p:    [batch, out_channels, height/2, width/2] (pooled feature map)
        """
       # extract data
        down = self.conv(x)
        
       # maxpooling size decrease
        p = self.pool(down)
        
        # down will be passed to the corresponding upsampling layer via "skip connection"  
        # p will continue to be passed to the next downsampling layer
        return down, p

UpSample 

upsamples the low-resolution feature map from the decoder by a factor of 2 using transposed convolution, concatenates it with the corresponding high-resolution skip-connected feature map from the encoder, and then applies a double convolution to fuse them into a refined feature map

In [ ]:
class UpSample(nn.Module):
    """
    upsample: ConvTranspose2d -> Concat -> DoubleConv
    
    Function:
    1. Use deconvolution (transposed convolution) to enlarge the feature map by a factor of 2.
    2. Concatenate the enlarged feature map with the feature map obtained through skip connection.
    3. Merge the concatenated feature maps using DoubleConv.     
    Core: The concatenation operation (Concat) is the essence of Unet!
    It combines the "details features during downsampling" and the "abstract features during upsampling".
    """
    def __init__(self, in_channels, out_channels):
        """
        Note:
        - The output channels of the deconvolution = in_channels // 2 (to match with the skip connections)
        - The total number of channels after concatenation = in_channels // 2 + in_channels // 2 = in_channels
        - Therefore, the input channels of DoubleConv = in_channels
        - The output channels of DoubleConv = out_channels
        """
        super(UpSample, self).__init__()
        
        # Deconvolution (transposed convolution) is used to enlarge the size.
        # With kernel_size=2 and stride=2, a 2-fold upsampling is achieved.
        # Result: 128x128 -> 256x256
        self.up = nn.ConvTranspose2d(
            in_channels,         
            in_channels // 2,     
            kernel_size=2,
            stride=2
        )
        
        # The concatenated feature maps are fused using DoubleConv
        # The input channels = in_channels (since the number of channels after concatenation = in_channels//2 + in_channels//2)
        # The output channels = out_channels
        self.conv = DoubleConv(in_channels, out_channels)
    
    def forward(self, x1, x2):
        """
        Forward propagation:
            x1: Feature map from below (upsampled input)
            x2: Feature map from the left encoder (skip connection)             
            Note: The sizes of x1 and x2 should be the same (after upsampling)
        """
        # Deconvolution amplification by a factor of x1 (dimension doubling)
        x1 = self.up(x1) #  x1 from feature map
        # size x1= size x2
        
        # 拼接 (在通道维度上拼接) Concatenation (concatenation along the channel dimension)
        # dim=1 表示在通道维度拼接
        x = torch.cat([x1, x2], dim=1) # x2 from downsmaple feature map
        # after combation: [batch, in_channels, height, width]
        
        return self.conv(x) # Immediately perform two 3x3 convolutions on the folded "mixed features"